In [3]:
import pandas as pd

# Load the original dataset
# Make sure the string matches your actual CSV file name
df = pd.read_csv('../data/netflix_titles.csv')

#Check the shape of the dataset
print(f"Dataset shape: {df.shape[0]} rows and {df.shape[1]} columns.")
print("Detected columns:")
print(df.columns.to_list())

#Display the first 3 rows to verify everything loaded correctly
df.head(3)


Dataset shape: 8807 rows and 12 columns.
Detected columns:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


Process ETL

In [4]:
# 1. Select the columns needed for Dim_Contenido
dim_contenido = df[['type', 'title', 'description', 'rating', 'duration', 'director', 'cast']].copy()

# 2. Drop duplicates based on title to ensure unique business keys
dim_contenido = dim_contenido.drop_duplicates(subset=['title']).reset_index(drop=True)

# 3. Create the Surrogate Key (sk_contenido) starting from 1
dim_contenido.insert(0, 'sk_contenido', dim_contenido.index + 1)

# 4. Rename 'cast' to 'cast_members' to match our physical model database design
dim_contenido = dim_contenido.rename(columns={'cast': 'cast_members'})

# 5. Preview the dimension table
print(f"Total unique shows in Dim_Contenido: {dim_contenido.shape[0]}")
dim_contenido.head(3)

Total unique shows in Dim_Contenido: 8807


,sk_contenido,type,title,description,rating,duration,director,cast_members
0,1,Movie,Dick Johnson Is Dead,"As her father nears the end of his life, filmm...",PG-13,90 min,Kirsten Johnson,NaN
1,2,TV Show,Blood & Water,"After crossing paths at a party, a Cape Town t...",TV-MA,2 Seasons,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban..."
2,3,TV Show,Ganglands,To protect his family from a powerful drug lor...,TV-MA,1 Season,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi..."


In [ ]:
# 1. Check which columns have missing values (NaN) in our dimension
print("Missing values before cleaning:")
print(dim_contenido.isnull().sum()) 
print("-" * 40)

# 2. Fill NaN values with meaningful placeholder strings
# We use 'Unknown' for categories or people, and 'Not Rated' for rating
cleaning_rules = {
    'director': 'Unknown Director',
    'cast_members': 'Unknown Cast',
    'rating': 'Not Rated',
    'duration': 'Unknown Duration'
}

dim_contenido = dim_contenido.fillna(value=cleaning_rules)

# 3. Verify that there are no more NaN values left
print("Missing values after cleaning:")
print(dim_contenido.isnull().sum())
print("-" * 40)

# 4. Preview a sample to see how the placeholder values look
dim_contenido.head(3)

Missing values before cleaning:
sk_contenido       0
type               0
title              0
description        0
rating             4
duration           3
director        2634
cast_members     825
dtype: int64
----------------------------------------
Missing values after cleaning:
sk_contenido    0
type            0
title           0
description     0
rating          0
duration        0
director        0
cast_members    0
dtype: int64
----------------------------------------


,sk_contenido,type,title,description,rating,duration,director,cast_members
0,1,Movie,Dick Johnson Is Dead,"As her father nears the end of his life, filmm...",PG-13,90 min,Kirsten Johnson,Unknown Cast
1,2,TV Show,Blood & Water,"After crossing paths at a party, a Cape Town t...",TV-MA,2 Seasons,Unknown Director,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban..."
2,3,TV Show,Ganglands,To protect his family from a powerful drug lor...,TV-MA,1 Season,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi..."


In [6]:
# 1. Ensure 'date_added' is in datetime format, handling errors
# We strip trailing spaces first just in case
df['date_added_clean'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

# 2. Extract unique combinations of original release_year and date_added components
# This prevents duplicating time rows unnecessarily
dim_tiempo = df[['release_year', 'date_added_clean']].copy()

# 3. Handle missing dates by filling with a placeholder or dropping 
# Since it's a small fraction, we drop rows with completely missing dates for the time table
dim_tiempo = dim_tiempo.dropna(subset=['date_added_clean']).drop_duplicates().reset_index(drop=True)

# 4. Extract analytical columns from the cleaned date
dim_tiempo['anio_added'] = dim_tiempo['date_added_clean'].dt.year.astype(int)
dim_tiempo['mes_added'] = dim_tiempo['date_added_clean'].dt.month.astype(int)
dim_tiempo['trimestre_added'] = dim_tiempo['date_added_clean'].dt.quarter.astype(int)

# 5. Create the Surrogate Key (sk_tiempo) starting from 1
dim_tiempo.insert(0, 'sk_tiempo', dim_tiempo.index + 1)

# 6. Rename column to match our schema design and preview
dim_tiempo = dim_tiempo.rename(columns={'date_added_clean': 'date_added'})
print(f"Total rows in Dim_Tiempo: {dim_tiempo.shape[0]}")
dim_tiempo.head(3)

Total rows in Dim_Tiempo: 4739


,sk_tiempo,release_year,date_added,anio_added,mes_added,trimestre_added
0,1,2020,2021-09-25,2021,9,3
1,2,2021,2021-09-24,2021,9,3
2,3,1993,2021-09-24,2021,9,3


In [7]:
# ==========================================
# 1. CREATING DIM_GEOGRAFIA (COUNTRIES)
# ==========================================

# Treat empty cells or NaN as 'Unknown Country' before splitting
df_country = df['country'].fillna('Unknown Country')

# Split rows that have multiple countries separated by commas and explode them into individual rows
countries_series = df_country.str.split(',').explode().str.strip()

# Keep unique countries and sort them alphabetically
unique_countries = sorted(countries_series.unique())

# Convert to DataFrame and insert the Surrogate Key (sk_geografia)
dim_geografia = pd.DataFrame(unique_countries, columns=['country'])
dim_geografia.insert(0, 'sk_geografia', dim_geografia.index + 1)


# ==========================================
# 2. CREATING DIM_CATEGORIAS (GENRES)
# ==========================================

# Treat empty cells or NaN as 'Uncategorized'
df_listed_in = df['listed_in'].fillna('Uncategorized')

# Split and explode multiple categories per show into individual items
categories_series = df_listed_in.str.split(',').explode().str.strip()

# Keep unique categories and sort them alphabetically
unique_categories = sorted(categories_series.unique())

# Convert to DataFrame and insert the Surrogate Key (sk_categoria)
dim_categorias = pd.DataFrame(unique_categories, columns=['listed_in'])
dim_categorias.insert(0, 'sk_categoria', dim_categorias.index + 1)


# ==========================================
# 3. PRINT PROGRESS AND PREVIEW
# ==========================================
print(f"Total rows in Dim_Geografia: {dim_geografia.shape[0]}")
print(f"Total rows in Dim_Categorias: {dim_categorias.shape[0]}\n")

print("--- Dim_Geografia Sample ---")
print(dim_geografia.head(3))
print("\n--- Dim_Categorias Sample ---")
print(dim_categorias.head(3))

Total rows in Dim_Geografia: 124
Total rows in Dim_Categorias: 42

--- Dim_Geografia Sample ---
   sk_geografia      country
0             1             
1             2  Afghanistan
2             3      Albania

--- Dim_Categorias Sample ---
   sk_categoria           listed_in
0             1  Action & Adventure
1             2      Anime Features
2             3        Anime Series


In [9]:
# 1. Prepare a base dataframe for the fact table mapping
fact_base = df.copy()

# 2. Map Dim_Contenido (matches on 'title' as it is our business key)
fact_shows = pd.merge(fact_base, dim_contenido[['sk_contenido', 'title']], on='title', how='left')

# 3. Clean and prepare the date column in the base to match Dim_Tiempo exactly
fact_shows['date_added_clean'] = pd.to_datetime(fact_shows['date_added'].str.strip(), errors='coerce')

# Map Dim_Tiempo (matches on original release_year and the cleaned date_added)
fact_shows = pd.merge(
    fact_shows, 
    dim_tiempo[['sk_tiempo', 'release_year', 'date_added']], 
    left_on=['release_year', 'date_added_clean'], 
    right_on=['release_year', 'date_added'], 
    how='left'
)

# 4. CORRECT WAY TO EXPLODE GEOGRAPHY:
# First fill NaN, then split into lists
fact_shows['country_clean'] = fact_shows['country'].fillna('Unknown Country').str.split(',')
# Explode the entire dataframe so rows multiply correctly, then strip whitespaces
fact_shows = fact_shows.explode('country_clean')
fact_shows['country_clean'] = fact_shows['country_clean'].str.strip()

# Map Dim_Geografia
fact_shows = pd.merge(fact_shows, dim_geografia, left_on='country_clean', right_on='country', how='left')


# 5. CORRECT WAY TO EXPLODE CATEGORIES:
# First fill NaN, then split into lists
fact_shows['listed_in_clean'] = fact_shows['listed_in'].fillna('Uncategorized').str.split(',')
# Explode the entire dataframe so rows multiply correctly, then strip whitespaces
fact_shows = fact_shows.explode('listed_in_clean')
fact_shows['listed_in_clean'] = fact_shows['listed_in_clean'].str.strip()

# Map Dim_Categorias
fact_shows = pd.merge(fact_shows, dim_categorias, left_on='listed_in_clean', right_on='listed_in', how='left')


# 6. Select and reorder only the final columns designed for Fact_Shows
final_fact_columns = ['show_id', 'sk_contenido', 'sk_tiempo', 'sk_geografia', 'sk_categoria']
fact_shows = fact_shows[final_fact_columns].reset_index(drop=True)

# 7. Fill any missing foreign key with a default integer if needed (e.g., if date was null)
fact_shows['sk_tiempo'] = fact_shows['sk_tiempo'].fillna(-1).astype(int)

# 8. Print progress and preview the star schema core
print(f"Total rows generated in Fact_Shows: {fact_shows.shape[0]}")
fact_shows.head(5)

Total rows generated in Fact_Shows: 23764


,show_id,sk_contenido,sk_tiempo,sk_geografia,sk_categoria
0,s1,1,1,117,11
1,s2,2,2,101,18
2,s2,2,2,101,35
3,s2,2,2,101,37
4,s3,3,2,118,9


Database Creation and Ingestion Code

In [15]:
from sqlalchemy import create_engine
import urllib

# 1. Configure the connection string for Microsoft SQL Server using Windows Authentication
# 'Trusted_Connection=yes' tells SQL Server to use your Windows login credentials
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=netflix_warehouse;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# 2. Create the SQL Server connection engine
engine_sqlserver = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

print("Starting data loading into the local Microsoft SQL Server...")

# 3. Ingest DataFrames into physical SQL Server tables
dim_contenido.to_sql('Dim_Contenido', con=engine_sqlserver, if_exists='replace', index=False)
dim_tiempo.to_sql('Dim_Tiempo', con=engine_sqlserver, if_exists='replace', index=False)
dim_geografia.to_sql('Dim_Geografia', con=engine_sqlserver, if_exists='replace', index=False)
dim_categorias.to_sql('Dim_Categorias', con=engine_sqlserver, if_exists='replace', index=False)
fact_shows.to_sql('Fact_Shows', con=engine_sqlserver, if_exists='replace', index=False)

print("ETL Ingestion successfully completed! Check your SQL Server Management Studio to see the tables.")

Starting data loading into the local Microsoft SQL Server...
ETL Ingestion successfully completed! Check your SQL Server Management Studio to see the tables.
